# دستیار هوشمند خرید و تحلیل محصولات — بخش دوم

این نوت‌بوک چهار قابلیت خواسته‌شده در بخش دوم پروژه را می‌سازد:

۱. جست‌وجو و کشف محصول با زبان طبیعی فارسی
۲. پرسش و پاسخ مبتنی بر نظرات کاربران، همراه با شناسه نظرات مورد استناد
۳. مقایسه محصولات
۴. تحلیل سطح دسته برای مدیران فروشگاه

ورودی این نوت‌بوک، دو جدول کاری است که `ali-01-EDA.ipynb` تولید کرده.

In [1]:
import gc
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

from dk_text import normalize_text, PLACEHOLDER_BRAND

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

DATA_DIR = Path.cwd()
ARTIFACT_DIR = DATA_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

# Catalogue prices are stored in rial; shoppers talk in toman.
RIAL_PER_TOMAN = 10

started = time.time()
products = pd.read_parquet(DATA_DIR / "products_work.parquet")
comments = pd.read_parquet(
    DATA_DIR / "comments_work.parquet",
    columns=["id", "product_id", "body", "body_norm", "title", "rate",
             "recommendation_status", "is_buyer", "likes", "body_len",
             "is_substantive", "advantages", "disadvantages", "created_at"],
)
print(f"loaded in {time.time() - started:.1f}s")
print(f"products: {len(products):>9,} x {products.shape[1]}")
print(f"comments: {len(comments):>9,} x {comments.shape[1]}")

# Same data contract the EDA notebook guaranteed.
assert products["id"].is_unique
assert comments["product_id"].isin(products["id"]).all()
print("data contract holds")

# product_id -> row positions, so evidence lookup is O(1) instead of a scan.
COMMENT_INDEX = comments.groupby("product_id").indices
PRODUCT_ROW = pd.Series(np.arange(len(products)), index=products["id"])
print(f"lookup tables built for {len(COMMENT_INDEX):,} products")

loaded in 0.7s
products:   331,599 x 33
comments: 6,153,060 x 14
data contract holds
lookup tables built for 331,599 products


## ۱. نمایه بازیابی محصول

از `TfidfVectorizer` استفاده می‌کنیم — همان ابزاری که در بخش سوم پروژه هم برای طبقه‌بندی
به کار رفته، تا کل پروژه روی یک پشته فنی بماند.

مدل embedding عمداً استفاده نشده: اندازه‌گیری نشان داد بازیابی واژه‌ای روی عبارات تمیز
محصول کیفیت خوبی دارد و هزینه‌اش کسری از بردارسازی ۳۳۱ هزار محصول است.

In [2]:
# max_df drops terms that appear in more than a quarter of the catalogue: they
# carry no discriminative power and pull every query towards the largest
# department (books, 28% of the catalogue).
started = time.time()
product_vectorizer = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.25, sublinear_tf=True,
)
PRODUCT_MATRIX = normalize(product_vectorizer.fit_transform(products["search_text"]))
print(f"index built in {time.time() - started:.1f}s")
print(f"shape: {PRODUCT_MATRIX.shape[0]:,} x {PRODUCT_MATRIX.shape[1]:,}")
print(f"non-zero entries: {PRODUCT_MATRIX.nnz:,} "
      f"({PRODUCT_MATRIX.data.nbytes / 1024**2:.0f} MB)")

# Latency check: one sparse matrix product against the whole catalogue.
probe = normalize(product_vectorizer.transform([normalize_text("کیف زنانه")]))
started = time.time()
for _ in range(20):
    _ = (PRODUCT_MATRIX @ probe.T).toarray().ravel()
print(f"query latency: {(time.time() - started) / 20 * 1000:.1f} ms")

index built in 5.9s
shape: 331,599 x 229,750
non-zero entries: 6,564,495 (50 MB)
query latency: 16.5 ms


## ۲. فهم پرس‌وجو — خط پایه قاعده‌محور

پرس‌وجوی محاوره‌ای فارسی سه چیز متفاوت در خود دارد که باید از هم جدا شوند:

| بخش جمله | نوع | مقصد |
|---|---|---|
| «کیف برای استفاده روزمره» | معنایی | بازیابی متنی |
| «خیلی گرون نباشه» | فیلتر عددی | شرط روی `Price` |
| «خریدارها راضی باشن» | فیلتر عددی | شرط روی `rec_score` |

اگر کل جمله را به بازیابی بدهیم، کلمات نیت مثل «می‌خوام» و «معرفی کن» با عنوان
کتاب‌ها تطبیق می‌خورند و نتیجه خراب می‌شود. این را در آزمایش دیدیم.

In [3]:
ZWNJ = "‌"
strip_zwnj = lambda s: s.replace(ZWNJ, "")

# Words that express intent rather than product. Removing them is what turns
# "یک کیف برای استفاده روزمره می‌خوام" into a query the retriever can use.
# Compared without ZWNJ, so "می‌خوام" and "میخوام" both match.
INTENT_WORDS = {strip_zwnj(normalize_text(w)) for w in """
و در به از که این را با های برای آن یک می بر است تا هم شد بود شده کند دارد
کنید شود نیز اما یا هر چه بی نه ای ها میخوام خوام میخواهم بخرم بخرید معرفی
کن کنید بگو پیشنهاد لطفا سلام چند چندتا یه سری بهترین بهتر خوب خوبی خوبه
عالی خیلی مناسب کاربردی استفاده روزمره وسیله وسایل چیزی چیزهایی دنبال میگردم
باشه باشد باشند باشن نباشه نباشد دارن دارند ازش ازشون شون راضی رضایت نظر نظرات
کاربران خریداران خریدارها درباره درموردش قیمت تومان تومن هزار
میلیون زیر بالای کمتر بیشتر حداکثر حداقل حدود تقریبا گرون گران ارزان ارزون
""".split()}

# Tokenise a query exactly the way the index does. scikit-learn's default
# pattern treats the ZWNJ as a boundary, so "استفاده‌ی" is indexed as
# "استفاده". Splitting the query on whitespace instead left the glued form
# intact, it missed the stopword list, and — because "استفاده" appears in only
# 39 product titles and therefore carries a huge IDF — every product that did
# contain it was launched to the top of the ranking.
QUERY_TOKEN = re.compile(r"\b\w\w+\b", re.UNICODE)

QUALITY_WORDS = ("راضی", "رضایت", "پیشنهاد", "محبوب", "پرفروش", "باکیفیت", "کیفیت")
PRICE_CONTEXT = ("تومان", "تومن", "قیمت", "ارزان", "ارزون", "گرون", "گران",
                 "بودجه", "هزار", "میلیون")
PRICE_PATTERN = re.compile(
    r"(?P<rel>زیر|کمتر از|حداکثر|بالای|بیشتر از|حداقل|حدود|تقریبا)?\s*"
    r"(?P<num>\d[\d,]*)\s*(?P<mult>هزار|میلیون)?\s*(?P<unit>تومان|تومن|ریال)?")
MULTIPLIERS = {"هزار": 1_000, "میلیون": 1_000_000}


def parse_price(text: str) -> tuple[int | None, int | None, str]:
    """Extract a (min_rial, max_rial) budget and return the text without it.

    A bare number is only treated as money when a unit or a multiplier backs
    it. Without that guard, "کودک ۵ ساله" would be read as a 5-rial ceiling
    and filter the entire catalogue away.
    """
    if not any(word in text for word in PRICE_CONTEXT):
        return None, None, text

    for match in PRICE_PATTERN.finditer(text):
        if not (match.group("unit") or match.group("mult")):
            continue
        amount = (int(match.group("num").replace(",", ""))
                  * MULTIPLIERS.get(match.group("mult"), 1) * RIAL_PER_TOMAN)
        rest = (text[:match.start()] + " " + text[match.end():]).strip()
        relation = match.group("rel")
        if relation in ("زیر", "کمتر از", "حداکثر"):
            return None, amount, rest
        if relation in ("بالای", "بیشتر از", "حداقل"):
            return amount, None, rest
        if relation in ("حدود", "تقریبا"):
            return int(amount * 0.7), int(amount * 1.3), rest
        return None, amount, rest

    # No explicit number: fall back on the vague wording shoppers actually use.
    if any(word in text for word in ("گرون نباشه", "گران نباشد", "ارزان", "ارزون")):
        return None, 500_000 * RIAL_PER_TOMAN, text
    return None, None, text


def understand_query_rules(query: str) -> dict:
    """Rule-based query understanding. This is the baseline, not the final system."""
    text = normalize_text(query)
    min_price, max_price, rest = parse_price(text)
    tokens = [t for t in QUERY_TOKEN.findall(rest)
              if strip_zwnj(t) not in INTENT_WORDS and not t.isdigit()]
    return {
        "concept": " ".join(tokens),
        "min_price": min_price,
        "max_price": max_price,
        "require_satisfaction": any(w in text for w in QUALITY_WORDS),
        "sub_category": None,
        "source": "rules",
    }


EXAMPLE_QUERIES = [
    "یک کیف برای استفاده روزمره می‌خوام که خیلی گرون نباشه و خریدارها هم ازش راضی باشن",
    "چند وسیله کاربردی برای سفر معرفی کن که نظر کاربران درباره‌شون خوب باشه",
    "کرم آبرسان صورت زیر ۲۰۰ هزار تومان",
    "یک اسباب بازی فکری برای کودک ۵ ساله",
]
for query in EXAMPLE_QUERIES:
    print(query)
    print("   ->", understand_query_rules(query))
    print()

یک کیف برای استفاده روزمره می‌خوام که خیلی گرون نباشه و خریدارها هم ازش راضی باشن
   -> {'concept': 'کیف', 'min_price': None, 'max_price': 5000000, 'require_satisfaction': True, 'sub_category': None, 'source': 'rules'}

چند وسیله کاربردی برای سفر معرفی کن که نظر کاربران درباره‌شون خوب باشه
   -> {'concept': 'سفر', 'min_price': None, 'max_price': None, 'require_satisfaction': False, 'sub_category': None, 'source': 'rules'}

کرم آبرسان صورت زیر ۲۰۰ هزار تومان
   -> {'concept': 'کرم آبرسان صورت', 'min_price': None, 'max_price': 2000000, 'require_satisfaction': False, 'sub_category': None, 'source': 'rules'}

یک اسباب بازی فکری برای کودک ۵ ساله
   -> {'concept': 'اسباب بازی فکری کودک ساله', 'min_price': None, 'max_price': None, 'require_satisfaction': False, 'sub_category': None, 'source': 'rules'}



## ۳. جست‌وجوی محصول — قابلیت ۱

سه مرحله پشت سر هم:

۱. شباهت متنی بین عبارت محصول و `search_text`
۲. فیلتر ساخت‌یافته روی قیمت و تعداد نظر
۳. رتبه‌بندی ترکیبی: شباهت متنی به‌علاوه `rec_score`

`rec_score` همان نمره منقبض‌شده فاز اول است. اگر به‌جایش `rec_ratio` خام را
می‌گذاشتیم، ۹۷ هزار محصول تک‌نظری با نمره کامل صدر جدول را می‌گرفتند.

In [4]:
SATISFACTION_WEIGHT = 0.15   # how much rec_score is allowed to move the ranking
MIN_REVIEWS_DEFAULT = 5      # below this a product has no usable satisfaction signal


def term_masks(tokens: list[str]) -> list[np.ndarray]:
    """For each query term, which products contain it at all."""
    masks = []
    for token in tokens:
        column = product_vectorizer.vocabulary_.get(token)
        if column is not None:
            masks.append(np.asarray((PRODUCT_MATRIX[:, column] > 0).todense()).ravel())
    return masks


def search_products(plan: dict, top_k: int = 8, min_reviews: int = MIN_REVIEWS_DEFAULT) -> pd.DataFrame:
    """Retrieve candidate products for an understood query.

    Cosine similarity alone is too forgiving: a product matching one rare
    query term outranks a product matching every common one. So terms are
    required to be present, not merely weighted — AND first, and if that is
    too strict to fill a page, the widest single term instead.
    """
    concept = plan.get("concept") or ""
    if not concept.strip():
        return products.head(0)

    query_vector = normalize(product_vectorizer.transform([normalize_text(concept)]))
    similarity = (PRODUCT_MATRIX @ query_vector.T).toarray().ravel()

    masks = term_masks(concept.split())
    if masks:
        all_terms = np.logical_and.reduce(masks)
        if all_terms.sum() >= top_k:
            similarity = np.where(all_terms, similarity, -np.inf)
        elif len(masks) > 1:
            widest = max(masks, key=lambda m: m.sum())
            if widest.sum() >= top_k:
                similarity = np.where(widest, similarity, -np.inf)

    keep = products["n_reviews"].to_numpy() >= min_reviews
    if plan.get("max_price"):
        keep &= products["Price"].to_numpy() <= plan["max_price"]
    if plan.get("min_price"):
        keep &= products["Price"].to_numpy() >= plan["min_price"]
    if plan.get("require_satisfaction"):
        keep &= products["evidence_tier"].isin(["moderate", "rich"]).to_numpy()
    if plan.get("sub_category"):
        keep &= (products["sub_category"] == plan["sub_category"]).to_numpy()

    score = similarity + SATISFACTION_WEIGHT * products["rec_score"].fillna(0).to_numpy()
    score = np.where(keep & np.isfinite(similarity) & (similarity > 0), score, -np.inf)

    n_hits = int(np.isfinite(score).sum())
    if n_hits == 0:
        return products.head(0)
    top = np.argpartition(-score, min(top_k, n_hits - 1))[:top_k]
    top = top[np.argsort(-score[top])]
    top = top[np.isfinite(score[top])]

    out = products.iloc[top].copy()
    out["similarity"] = similarity[top]
    out["score"] = score[top]
    out["price_toman"] = out["Price"] // RIAL_PER_TOMAN
    return out


def show(frame: pd.DataFrame) -> None:
    """Compact console view of a result set."""
    if frame.empty:
        print("  (no match)")
        return
    for _, row in frame.iterrows():
        print(f"  {row['score']:.3f} | {row['price_toman']:>9,}T | n={row['n_reviews']:>3} "
              f"| rec={row['rec_score']:.2f} | {row['evidence_tier']:<8} | {row['title_fa'][:52]}")


for query in EXAMPLE_QUERIES:
    plan = understand_query_rules(query)
    started = time.time()
    results = search_products(plan)
    print(f"\n{query}")
    print(f"  concept={plan['concept']!r}  max_price={plan['max_price']}  "
          f"({(time.time() - started) * 1000:.0f} ms)")
    show(results.head(5))


یک کیف برای استفاده روزمره می‌خوام که خیلی گرون نباشه و خریدارها هم ازش راضی باشن
  concept='کیف'  max_price=5000000  (111 ms)
  0.598 |   209,000T | n=200 | rec=0.88 | rich     | کیف زنانه کد 431405
  0.549 |    98,000T | n=200 | rec=0.82 | rich     | کیف رودوشی زنانه مدل SH1113
  0.547 |   295,000T | n= 28 | rec=0.80 | rich     | کیف رودوشی زنانه مدل 2782
  0.546 |   146,000T | n= 16 | rec=0.91 | moderate | کیف رودوشی مردانه مدل BR8
  0.543 |   139,000T | n= 20 | rec=0.89 | rich     |  کیف رودوشی مردانه مدل HE126

چند وسیله کاربردی برای سفر معرفی کن که نظر کاربران درباره‌شون خوب باشه
  concept='سفر'  max_price=None  (63 ms)
  0.431 |    50,000T | n=  8 | rec=0.87 | thin     | کتاب راهنمای سفر ایران اثر وحید رضا اخباری انتشارات 
  0.427 |   175,000T | n=200 | rec=0.88 | rich     | کیف کمری رجینال مدل RS17
  0.422 |    40,000T | n= 12 | rec=0.76 | moderate | کتاب Iran اثر lonely planet انتشارات راهنمای سفر
  0.420 |   102,000T | n= 15 | rec=0.91 | thin     | ساک ورزشی مدل fi2000f
  0.

## ۴. اتصال به مدل زبانی با حسابداری هزینه

دسترسی مستقیم به `generativelanguage.googleapis.com` از ایران مسدود است، پس از
درگاه **متیس** استفاده می‌کنیم که رابطی سازگار با `OpenAI` روی همان مدل‌های
جمینای ارائه می‌دهد:

```
https://api.metisai.ir/api/v1/wrapper/google
```

مدل‌ها عوض نشده‌اند — همان زنجیره `gemini-3.5-flash` و پشتیبان‌هایش.

سه چیز از روز اول کنترل می‌شود چون بودجه محدود است:

- **شمارش دقیق توکن** روی هر فراخوانی و ثبت در یک لاگ
- **کش پاسخ** روی دیسک، تا اجرای دوباره نوت‌بوک هزینه‌ای نداشته باشد
- **تعداد فراخوانی** که در گزارش نهایی می‌آید

In [5]:
# Put the key in a .env file next to this notebook: METIS_API_KEY=...
import os

import openai

try:
    from dotenv import load_dotenv
    load_dotenv(DATA_DIR / ".env")
except ImportError:
    print("python-dotenv not installed; falling back to the environment")

# The gateway speaks the OpenAI protocol, so the standard client works with
# nothing but a different base URL. api.tapsage.com is the mirror to use from
# hosts that cannot resolve the .ir domain, such as Colab.
METIS_BASE_URL = "https://api.metisai.ir/api/v1/wrapper/google"
API_KEY = os.environ.get("METIS_API_KEY", "").strip()
print(f"key present: {bool(API_KEY)} | endpoint: {METIS_BASE_URL}")

client = None
LIVE_API = False
if API_KEY:
    client = openai.OpenAI(base_url=METIS_BASE_URL, api_key=API_KEY, timeout=120.0)
    try:
        catalogue = sorted(m.id for m in client.models.list().data)
        LIVE_API = True
        flash = [m for m in catalogue
                 if "flash" in m and "image" not in m and "tts" not in m]
        print(f"\n{len(catalogue)} models reachable; the flash family:")
        for name in flash:
            print("  ", name)
    except Exception as error:
        print(f"\ncould not reach the gateway: {type(error).__name__} — {str(error)[:120]}")

if not LIVE_API:
    print("\nRunning in cache-only mode: every answer must already be in")
    print("artifacts/llm_cache.json. Nothing below will call the API.")

key present: True | endpoint: https://api.metisai.ir/api/v1/wrapper/google

44 models reachable; the flash family:
   gemini-1.5-flash
   gemini-1.5-flash-8b
   gemini-2.0-flash
   gemini-2.0-flash-lite-preview
   gemini-2.5-flash
   gemini-2.5-flash-lite
   gemini-2.5-flash-lite-preview
   gemini-2.5-flash-lite-preview-06-17
   gemini-2.5-flash-preview
   gemini-2.5-flash-preview-04-17
   gemini-2.5-flash-preview-05-20
   gemini-3-flash-preview
   gemini-3.1-flash-lite
   gemini-3.1-flash-lite-preview
   gemini-3.5-flash
   gemini-3.5-flash-lite
   gemini-3.6-flash
   gemini-3.7-flash


In [6]:
# One model id is a single point of failure: quota and availability are both
# per model. The chain is tried in order, quality first, then progressively
# cheaper. This doubles as the multi-model router.
MODEL_CHAIN = ["gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
MODEL = MODEL_CHAIN[0]

# Fill these in from the Gemini pricing page. Token counts below are exact
# regardless; only the money figure depends on these two numbers.
USD_PER_1M_INPUT = 0.30
USD_PER_1M_OUTPUT = 2.50

CACHE_PATH = ARTIFACT_DIR / "llm_cache.json"
USAGE_PATH = ARTIFACT_DIR / "llm_usage.json"
CACHE = json.loads(CACHE_PATH.read_text(encoding="utf-8")) if CACHE_PATH.exists() else {}
USAGE = json.loads(USAGE_PATH.read_text(encoding="utf-8")) if USAGE_PATH.exists() else []
print(f"cache: {len(CACHE)} entries | usage log: {len(USAGE)} calls")


def ask_gemini(prompt: str, *, tag: str, as_json: bool = False,
               temperature: float = 0.2, max_tokens: int = 1024) -> str:
    """Call Gemini once, with an on-disk cache and per-call token accounting.

    `tag` labels the call site so the cost report can be broken down by
    capability rather than being one undifferentiated total.
    """
    # The cache key deliberately omits the model: an answer already paid for
    # stays valid no matter which model in the chain produced it.
    key = f"{tag}|{as_json}|{prompt}"
    if key in CACHE:
        return CACHE[key]
    if not LIVE_API:
        raise RuntimeError(
            "No cached answer and the API is unreachable from this network. "
            "Run this notebook on Colab to populate artifacts/llm_cache.json, "
            "then copy the file back here.")

    def complete(model_name: str, limit: int):
        # Generation-3 models bill their chain of thought against the same
        # output budget. Left on, a 400-token cap produced 396 tokens of
        # thinking and 30 characters of answer. The gateway forwards
        # thinking_config verbatim, so switching it off costs one extra field.
        # response_format is deliberately absent: the gateway rejects it, so
        # JSON is requested in the prompt and parsed defensively by the caller.
        return client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=limit,
            temperature=temperature,
            extra_body={"thinking_config": {"thinking_budget": 0}},
        )

    response, used_model, elapsed, limit = None, None, 0.0, max_tokens
    for candidate in MODEL_CHAIN:
        for attempt in range(2):
            try:
                started = time.time()
                response = complete(candidate, limit)
                elapsed, used_model = time.time() - started, candidate
                break
            except Exception as error:
                message = str(error)
                if "429" not in message and "rate" not in message.lower():
                    raise
                delay = re.search(r"retry in ([\d.]+)s", message)
                wait = min(float(delay.group(1)) + 1, 30) if delay else 5 * (attempt + 1)
                print(f"  rate limited on {candidate}; waiting {wait:.0f}s")
                time.sleep(wait)
        if response is not None:
            break
        print(f"  {candidate} unavailable -> falling back")
    if response is None:
        raise RuntimeError("every model in MODEL_CHAIN refused the request")

    # Retry once at a larger budget rather than keeping a half-written answer.
    if response.choices[0].finish_reason == "length" and limit < 8192:
        limit = min(limit * 3, 8192)
        print(f"  answer hit the token cap; retrying with max_tokens={limit}")
        started = time.time()
        response = complete(used_model, limit)
        elapsed = time.time() - started

    usage = response.usage
    record = {
        "tag": tag,
        "model": used_model,
        "input_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
        "output_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
        "thought_tokens": 0,          # the gateway does not report them separately
        "latency_s": round(elapsed, 3),
    }
    record["usd"] = round(
        record["input_tokens"] / 1e6 * USD_PER_1M_INPUT
        + (record["output_tokens"] + record["thought_tokens"]) / 1e6 * USD_PER_1M_OUTPUT, 6)
    USAGE.append(record)
    USAGE_PATH.write_text(json.dumps(USAGE, ensure_ascii=False, indent=1), encoding="utf-8")

    text = (response.choices[0].message.content or "").strip()
    if not text:
        raise RuntimeError(f"{used_model} returned an empty answer "
                           f"(finish_reason={response.choices[0].finish_reason})")
    # Never cache a malformed answer: a cached "Here is the JSON requested:"
    # with no object behind it silently poisons every later run.
    if as_json and "{" not in text:
        raise RuntimeError(f"{used_model} was asked for JSON and returned none: {text[:80]!r}")
    CACHE[key] = text
    CACHE_PATH.write_text(json.dumps(CACHE, ensure_ascii=False), encoding="utf-8")
    return text


def cost_report() -> pd.DataFrame:
    """Token and cost totals, broken down by call site."""
    if not USAGE:
        return pd.DataFrame()
    frame = pd.DataFrame(USAGE)
    if "thought_tokens" not in frame:
        frame["thought_tokens"] = 0
    frame["thought_tokens"] = frame["thought_tokens"].fillna(0)
    report = frame.groupby("tag").agg(
        calls=("tag", "size"),
        input_tokens=("input_tokens", "sum"),
        output_tokens=("output_tokens", "sum"),
        thought_tokens=("thought_tokens", "sum"),
        usd=("usd", "sum"),
        mean_latency_s=("latency_s", "mean"),
    )
    report.loc["TOTAL"] = report.sum(numeric_only=True)
    report.loc["TOTAL", "mean_latency_s"] = frame["latency_s"].mean()
    return report.round(4)


if LIVE_API or CACHE:
    reply = ask_gemini("سلام. فقط بنویس: اتصال برقرار است.", tag="smoke-test", max_tokens=200)
    print("reply:", reply.strip())
else:
    print("skipped: no live API and no cache yet")
print()
report = cost_report()
print(report.to_string() if not report.empty else "(no calls recorded yet)")

cache: 32 entries | usage log: 56 calls
reply: 

                     calls  input_tokens  output_tokens  thought_tokens     usd  mean_latency_s
tag                                                                                            
category-analytics     7.0        7837.0         2732.0             0.0  0.0092          4.3056
comparison             7.0       18692.0         2401.0             0.0  0.0116          5.6200
product-qa            23.0       34525.0         4948.0             0.0  0.0227          8.3573
query-understanding   17.0        3722.0          468.0             0.0  0.0023          6.2526
smoke-test             2.0          24.0            1.0             0.0  0.0000          8.8695
TOTAL                 56.0       64800.0        10550.0             0.0  0.0458          6.8881


## ۵. فهم پرس‌وجو با مدل زبانی

نسخه بهبودیافته لایه ۱. همان کاری را می‌کند که نسخه قاعده‌محور می‌کرد، ولی
عبارات محاوره‌ای مبهم مثل «خیلی گرون نباشه» را بهتر می‌فهمد.

خروجی ساخت‌یافته `JSON` است تا مستقیماً به موتور بازیابی داده شود. در ادامه هر دو
نسخه را روی همان پرس‌وجوها اجرا می‌کنیم تا اختلافشان دیده شود.

In [7]:
SUB_CATEGORIES = sorted(products["sub_category"].unique().tolist())

QUERY_PROMPT = """تو دستیار یک فروشگاه اینترنتی هستی. جمله کاربر را به یک شیء JSON تبدیل کن.

فیلدها:
- concept: فقط نام و ویژگی خود کالا به فارسی، بدون فعل و بدون کلمات نیت. مثال: "کیف دوشی زنانه"
- max_price_toman: حداکثر قیمت به تومان، یا null
- min_price_toman: حداقل قیمت به تومان، یا null
- require_satisfaction: true اگر کاربر به رضایت خریداران اشاره کرده
- sub_category: یکی از {categories} یا null

راهنمای قیمت: اگر کاربر گفت «گران نباشد» یا «ارزان» و عددی نگفت، max_price_toman را 500000 بگذار.

فقط JSON خروجی بده.

جمله کاربر: {query}"""


def understand_query_llm(query: str) -> dict:
    """LLM query understanding; falls back to the rule-based plan on any failure."""
    prompt = QUERY_PROMPT.format(categories=SUB_CATEGORIES, query=query)
    raw = ask_gemini(prompt, tag="query-understanding", as_json=True,
                     temperature=0.0, max_tokens=256)
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, re.S)
        if not match:
            return understand_query_rules(query)
        parsed = json.loads(match.group())

    to_rial = lambda v: int(v) * RIAL_PER_TOMAN if v else None
    return {
        "concept": parsed.get("concept") or "",
        "min_price": to_rial(parsed.get("min_price_toman")),
        "max_price": to_rial(parsed.get("max_price_toman")),
        "require_satisfaction": bool(parsed.get("require_satisfaction")),
        "sub_category": parsed.get("sub_category"),
        "source": "llm",
    }


for query in EXAMPLE_QUERIES:
    print(f"\n=== {query} ===")
    for name, plan in (("rules", understand_query_rules(query)),
                       ("llm  ", understand_query_llm(query))):
        print(f"  [{name}] concept={plan['concept']!r} max={plan.get('max_price')} "
              f"sat={plan.get('require_satisfaction')} cat={plan.get('sub_category')}")
        show(search_products(plan).head(3))


=== یک کیف برای استفاده روزمره می‌خوام که خیلی گرون نباشه و خریدارها هم ازش راضی باشن ===
  [rules] concept='کیف' max=5000000 sat=True cat=None
  0.598 |   209,000T | n=200 | rec=0.88 | rich     | کیف زنانه کد 431405
  0.549 |    98,000T | n=200 | rec=0.82 | rich     | کیف رودوشی زنانه مدل SH1113
  0.547 |   295,000T | n= 28 | rec=0.80 | rich     | کیف رودوشی زنانه مدل 2782
  [llm  ] concept='کیف روزمره' max=5000000 sat=True cat=clothe
  0.423 |   209,000T | n=200 | rec=0.88 | rich     | کیف زنانه کد 431405
  0.392 |   146,000T | n= 16 | rec=0.91 | moderate | کیف رودوشی مردانه مدل BR8
  0.390 |   121,840T | n= 58 | rec=0.90 | rich     | کیف رودوشی مدل GF20

=== چند وسیله کاربردی برای سفر معرفی کن که نظر کاربران درباره‌شون خوب باشه ===
  [rules] concept='سفر' max=None sat=False cat=None
  0.431 |    50,000T | n=  8 | rec=0.87 | thin     | کتاب راهنمای سفر ایران اثر وحید رضا اخباری انتشارات 
  0.427 |   175,000T | n=200 | rec=0.88 | rich     | کیف کمری رجینال مدل RS17
  0.422 |    40,00

## ۶. بازیابی شواهد و پرسش و پاسخ — قابلیت ۲

محصول از قبل مشخص است، پس نیازی به نمایه سراسری نظرات نیست: نظرات همان محصول
برداشته می‌شوند و نسبت به سؤال رتبه‌بندی می‌شوند.

پاسخ **حتماً** شناسه نظرات مورد استناد را برمی‌گرداند، و سطح ادعای مجاز با
`evidence_tier` کنترل می‌شود.

In [8]:
TIER_INSTRUCTION = {
    "rich": "شواهد کافی است. می‌توانی الگوهای پرتکرار را گزارش کنی.",
    "moderate": "شواهد متوسط است. الگو را بگو ولی صریحاً بنویس که بر پایه تعداد محدودی نظر است.",
    "thin": "شواهد کم است. فقط نظرات را نقل کن و هیچ الگوی کلی نتیجه نگیر.",
    "none": "هیچ نظر قابل استنادی وجود ندارد. اعلام کن که نمی‌توانی درباره تجربه کاربران نظر بدهی.",
}


def get_evidence(product_id: int, question: str, top_k: int = 20,
                 substantive_only: bool = True) -> pd.DataFrame:
    """Rank one product's reviews against a question.

    A per-product TF-IDF is cheap here: the median product carries 15 reviews
    and the maximum is 233, so this fits comfortably inside one request.
    """
    positions = COMMENT_INDEX.get(product_id)
    if positions is None or len(positions) == 0:
        return comments.head(0)
    pool = comments.iloc[positions]
    if substantive_only and pool["is_substantive"].any():
        pool = pool[pool["is_substantive"]]
    if len(pool) <= top_k:
        return pool.copy()

    local = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    matrix = normalize(local.fit_transform(pool["body_norm"].fillna("")))
    query_vector = normalize(local.transform([normalize_text(question)]))
    similarity = (matrix @ query_vector.T).toarray().ravel()

    # Longer reviews break ties: they carry more for the model to work with.
    order = np.lexsort((-pool["body_len"].to_numpy(), -similarity))[:top_k]
    out = pool.iloc[order].copy()
    out["relevance"] = similarity[order]
    return out


def product_facts(product_id: int) -> dict:
    """Structured catalogue facts, kept separate from review evidence."""
    row = products.loc[PRODUCT_ROW[product_id]]
    return {
        "id": int(row["id"]),
        "title": row["title_fa"],
        "brand": None if row["Brand"] == PLACEHOLDER_BRAND else row["Brand"],
        "category": row["Category1"],
        "price_toman": int(row["Price"]) // RIAL_PER_TOMAN,
        "n_reviews": int(row["n_reviews"]),
        "n_substantive": int(row["n_substantive"]),
        "rec_ratio": None if pd.isna(row["rec_ratio"]) else round(float(row["rec_ratio"]), 3),
        "rec_score": round(float(row["rec_score"]), 3),
        "mean_rate": None if pd.isna(row["mean_rate"]) else round(float(row["mean_rate"]), 2),
        "evidence_tier": str(row["evidence_tier"]),
    }


demo_id = int(products.sort_values("n_substantive", ascending=False).iloc[3]["id"])
print(json.dumps(product_facts(demo_id), ensure_ascii=False, indent=1))
sample = get_evidence(demo_id, "ایرادهای پرتکرار این محصول چیست؟", top_k=5)
print(f"\nevidence rows: {len(sample)}")
for _, row in sample.iterrows():
    print(f"  [{row['id']}] {str(row['body'])[:80]}")

{
 "id": 214936,
 "title": "ست اصلاح فیلیپس مدل QG3380/16",
 "brand": "فیلیپس",
 "category": "اصلاح موی صورت",
 "price_toman": 25140000,
 "n_reviews": 200,
 "n_substantive": 161,
 "rec_ratio": 0.833,
 "rec_score": 0.83,
 "mean_rate": 3.81,
 "evidence_tier": "rich"
}

evidence rows: 5
  [43131528] من این محصول رو ۵ سال پیش به قیمت ۲۶۳ هزار تومان خریدم ! خوبه ولی با این قیمت اص
  [1758539] من این محصول رو استفاده کردم.از تمامی ابزار اصلاح صورت و بدن و بینی استفاده کردم
  [4731114] من این محصول رو۲ سال پیشخریدم و واقعا ازش راضی هستم ، هرچیزی که از یه ریش تراش ا
  [8940810] سال ۹۶ این محصول رو خریداری کردم باید بگم که واقعا یدون ایراد و نقص هست پیشنهاد 
  [7038654] موتور دستگاه توان لازم برای یک اصلاح خوب رو نداره
شاید برند فیلیپس در تولید این


In [9]:
ANSWER_PROMPT = """تو دستیار خرید یک فروشگاه اینترنتی هستی و فقط بر پایه شواهد زیر پاسخ می‌دهی.

## اطلاعات محصول (داده مستقیم فروشگاه)
{facts}

## نظرات کاربران (شواهد)
{evidence}

## قواعد
- {tier_rule}
- هر ادعایی درباره تجربه کاربران را با شناسه نظر داخل کروشه مستند کن، مثل [12345].
- بین سه چیز تفکیک قائل شو: داده مستقیم فروشگاه، شواهد نظرات، و استنتاج خودت.
- چیزی که در شواهد نیست را به عنوان واقعیت بیان نکن.
- پاسخ فارسی، حداکثر ۱۵۰ کلمه.

## سؤال کاربر
{question}"""


# Models group several ids inside one bracket, separated by an ASCII or a
# Persian comma: [1758539، 9385022]. Matching only single-id brackets
# under-reports grounding badly, so capture the group and split it.
CITATION_PATTERN = re.compile(r"\[([\d\s,،]+)\]")


def extract_citations(answer: str) -> list[int]:
    """Pull every review id out of an answer, including grouped citations."""
    found = set()
    for group in CITATION_PATTERN.findall(answer):
        found.update(int(x) for x in re.findall(r"\d{4,}", group))
    return sorted(found)


def format_evidence(evidence: pd.DataFrame) -> str:
    """Render reviews as numbered, citable lines."""
    lines = []
    for _, row in evidence.iterrows():
        body = str(row["body"]).replace("\n", " ").strip()
        extra = ""
        if pd.notna(row["disadvantages"]):
            extra += f" | منفی: {row['disadvantages']}"
        if pd.notna(row["advantages"]):
            extra += f" | مثبت: {row['advantages']}"
        lines.append(f"[{row['id']}] (امتیاز {row['rate']}) {body}{extra}")
    return "\n".join(lines)


def answer_about_product(product_id: int, question: str, top_k: int = 20) -> dict:
    """Capability 2: answer a question about one product, grounded in its reviews."""
    facts = product_facts(product_id)
    evidence = get_evidence(product_id, question, top_k=top_k)
    started = time.time()
    prompt = ANSWER_PROMPT.format(
        facts=json.dumps(facts, ensure_ascii=False, indent=1),
        evidence=format_evidence(evidence) or "(بدون نظر)",
        tier_rule=TIER_INSTRUCTION[facts["evidence_tier"]],
        question=question,
    )
    # Generation-3 models spend part of the output budget on internal
    # reasoning, so a tight cap truncates the answer mid-sentence.
    answer = ask_gemini(prompt, tag="product-qa", max_tokens=1400)
    return {
        "question": question,
        "product": facts["title"],
        "evidence_tier": facts["evidence_tier"],
        "answer": answer.strip(),
        "evidence_ids": evidence["id"].tolist(),
        "cited_ids": extract_citations(answer),
        "latency_s": round(time.time() - started, 2),
    }


for question in ["مردم بیشتر از چه چیزی در این محصول راضی بودند؟",
                 "ایرادهای پرتکرار این محصول چیست؟"]:
    result = answer_about_product(demo_id, question)
    print(f"\n{'='*70}\nسؤال: {result['question']}")
    print(f"محصول: {result['product']}  |  tier={result['evidence_tier']}")
    print(f"{'='*70}\n{result['answer']}")
    print(f"\nشواهد داده‌شده: {len(result['evidence_ids'])} | "
          f"استناد‌شده: {len(result['cited_ids'])} -> {result['cited_ids'][:8]}")


سؤال: مردم بیشتر از چه چیزی در این محصول راضی بودند؟
محصول: ست اصلاح فیلیپس مدل QG3380/16  |  tier=rich
بر اساس شواهد نظرات کاربران، موارد زیر بیشترین میزان رضایت را در میان خریداران داشته‌اند:

* **سری‌های متنوع و کاربردی:** بسیاری از کاربران به داشتن سری‌های متعدد برای اصلاح صورت، بدن، موی بینی و گوش اشاره کرده و آن‌ها را کاربردی و دقیق دانسته‌اند [1758539، 9385022، 1714207].
* **عملکرد باتری و شارژ سریع:** نگهداری خوب شارژ و قابلیت شارژ سریع از نقاط قوت پرتکرار در نظرات است [6568903، 1978844، 1691750].
* **خاصیت ضدآب بودن:** قابلیت استفاده زیر دوش و ضدآب بودن دستگاه مورد رضایت کاربران بوده است [6568903، 1758539، 9385022].
* **کم‌صدا و سبک بودن:** طراحی سبک، کم‌صدا و نرم بودن تیغه اصلی از دیگر موارد مثبت ذکر شده است [1710496، 1714207].

طبق داده‌های مستقیم فروشگاه، این محصول با میانگین امتیاز ۳.۸۱ از ۵ و ضریب رضایت حدود ۸۳٪، توانسته نظر موافق اکثر خریداران را جلب کند.

شواهد داده‌شده: 20 | استناد‌شده: 7 -> [1691750, 1710496, 1714207, 1758539, 1978844, 6568903, 9385022]

سؤال: ایراده

## ۷. مقایسه محصولات — قابلیت ۳

برای هر محصول، حقایق ساخت‌یافته و شواهد نظرات جداگانه جمع می‌شود و مدل موظف است
تفاوت میان داده مستقیم، شواهد کاربران و استنتاج خودش را حفظ کند.

In [10]:
COMPARE_PROMPT = """دو یا چند محصول زیر را برای یک خریدار مقایسه کن.

{blocks}

## قواعد
- ابتدا تفاوت‌های عددی قابل استناد را بگو (قیمت، تعداد نظر، درصد پیشنهاد خرید).
- سپس تفاوت تجربه کاربران را با شناسه نظر مستند کن، مثل [12345].
- اگر شواهد یکی از محصولات کم است، صریح بگو که مقایسه نامتقارن است.
- در پایان یک پیشنهاد بده و روشن کن که استنتاج توست نه واقعیت داده.
- پاسخ فارسی، حداکثر ۲۰۰ کلمه."""


def compare_products(product_ids: list[int], question: str = "کدام انتخاب بهتری است و چرا؟",
                     per_product: int = 10) -> dict:
    """Capability 3: compare several products on data and on user experience."""
    blocks, evidence_ids = [], {}
    for pid in product_ids:
        facts = product_facts(pid)
        evidence = get_evidence(pid, question, top_k=per_product)
        evidence_ids[pid] = evidence["id"].tolist()
        blocks.append(
            f"## محصول {facts['title']}\n"
            f"### داده فروشگاه\n{json.dumps(facts, ensure_ascii=False, indent=1)}\n"
            f"### نظرات\n{format_evidence(evidence) or '(بدون نظر)'}"
        )
    started = time.time()
    answer = ask_gemini(COMPARE_PROMPT.format(blocks="\n\n".join(blocks)),
                        tag="comparison", max_tokens=2048)
    return {
        "product_ids": product_ids,
        "answer": answer.strip(),
        "evidence_ids": evidence_ids,
        "cited_ids": extract_citations(answer),
        "latency_s": round(time.time() - started, 2),
    }


# Two comparable products: same category, both with rich evidence.
candidates = products[(products["Category1"] == products.loc[PRODUCT_ROW[demo_id], "Category1"])
                      & (products["evidence_tier"] == "rich")]
pair = candidates.nlargest(2, "n_substantive")["id"].tolist()
result = compare_products(pair)
for pid in pair:
    print(f"- {product_facts(pid)['title'][:70]}")
print(f"\n{'='*70}\n{result['answer']}")
print(f"\nاستناد‌شده: {result['cited_ids'][:10]}  |  {result['latency_s']}s")

- ست اصلاح فیلیپس مدل QG3380/16
- ماشین اصلاح صورت فیلیپس مدل S9031

برای مقایسه دو محصول **ست اصلاح فیلیپس QG3380/16** و **ماشین اصلاح صورت فیلیپس S9031**، ابتدا به داده‌های فروشگاهی نگاهی می‌اندازیم:

* **قیمت:** مدل QG3380/16 حدود ۲۵.۱ میلیون تومان و مدل S9031 حدود ۱۴.۴ میلیون تومان است.
* **تعداد نظرات:** مدل اول ۲۰۰ نظر و مدل دوم ۱۸۶ نظر ثبت‌شده دارند که نشان‌دهنده حجم نمونه نزدیک به هم و مقایسه متقارن است.
* **درصد پیشنهاد خرید (rec_ratio):** مدل S9031 با حدود ۸۸.۹٪ رضایت، محبوب‌تر از مدل QG3380/16 با حدود ۸۳.۳٪ است. میانگین امتیاز آن‌ها نیز به ترتیب ۴.۰۵ و ۳.۸۱ است.

**تجربه کاربران:**
* **مدل S9031 (مخصوص تراشیدن کامل):** کاربران آن را بسیار کم‌صدا و عالی برای ته‌نشین کردن صفر صورت می‌دانند [1821883]. این دستگاه بدون ایجاد زخم و حساسیت پوست را کاملاً صاف می‌کند [1540309]، هرچند برخی به ضعیف بودن سری خط‌زن آن اشاره کرده‌اند [1821883].
* **مدل QG3380/16 (ست همه‌کاره):** این مدل برای فرم‌دهی ریش و اصلاح بدن مناسب است و سری‌های متنوعی دارد [1691750]، اما موها را کاملاً صفر نمی‌زند 

## ۸. تحلیل سطح دسته — قابلیت ۴

بیشتر این بخش **بدون مدل زبانی** انجام می‌شود. ستون‌های `advantages` و
`disadvantages` را خود کاربران پر کرده‌اند، پس شکایت‌های پرتکرار با شمارش
مستقیم قابل استخراج‌اند و توکنی مصرف نمی‌کنند.

In [11]:
# Shoppers write "ندارد" in the negatives field to mean "it has no downside".
# Left in, these non-complaints top the ranking and invert its meaning: the
# first run reported "ندارد" as the most frequent complaint, 429 times.
NON_COMPLAINTS = {normalize_text(w) for w in (
    "ندارد", "نداره", "نداشت", "نداشتم", "ندیدم", "ندارم", "هیچی", "هیچ",
    "نبود", "فعلا ندارد", "فعلا ندیدم", "هنوز ندیدم", "تا الان ندیدم",
    "چیزی ندیدم", "موردی ندارد", "مورد خاصی نبود", "نمیدونم", "نمی دانم",
    "نه", "خیر", "-", "_", "ن")}

# Negation and intensity words carry no topic. Dropping them before the
# word-level pass is what lets "صفر زن نیست" and "صفر نمیزنه" meet.
FILLER_WORDS = {normalize_text(w) for w in """
نیست نداره ندارد نمیزنه نمیزند کم ضعیف بد پایین بالا زیاد خیلی نسبتا
یکم کمی است هست بودن شدن این آن که را از به با در و یا
""".split()}


def _greedy_group(freq: np.ndarray, matrix, threshold: float) -> list[np.ndarray]:
    """Group rows of a normalised matrix, most frequent item first.

    Greedy rather than a full clustering algorithm on purpose: the most
    frequent phrase becomes the label of its group, which keeps the output
    readable and is deterministic.
    """
    assigned = np.full(matrix.shape[0], -1)
    groups = []
    for i in np.argsort(-freq):
        if assigned[i] != -1:
            continue
        similarity = (matrix @ matrix[i].T).toarray().ravel()
        members = np.where((similarity >= threshold) & (assigned == -1))[0]
        members = np.union1d(members, [i])   # an all-zero row still seeds its own group
        assigned[members] = len(groups)
        groups.append(members)
    return groups


def cluster_complaints(counts: pd.Series, char_threshold: float = 0.55,
                       word_threshold: float = 0.65,
                       max_phrases: int = 6000) -> pd.DataFrame:
    """Merge complaint phrases that mean the same thing.

    Raw counting fragments one complaint across many spellings, so the true
    leader is hidden. Two passes fix two different problems:

    1. character n-grams merge spelling and morphology variants
       ("باتری ضعیف" / "باطری ضعیف", "صدای زیاد" / "صدای نسبتا زیاد")
    2. content words merge different phrasings of the same concept
       ("صفر زن نیست" / "صفر نمیزنه")
    """
    # Grouping is quadratic in the number of phrases, and a department-wide
    # scope carries ~71k of them. The tail is one-off wordings that cannot
    # lead the ranking, so only the frequent head is clustered.
    counts = counts.head(max_phrases)
    phrases, freq = counts.index.to_numpy(), counts.to_numpy()
    if len(phrases) < 2:
        return pd.DataFrame({"complaint": phrases, "count": freq, "variants": 1})

    char_matrix = normalize(TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 4)).fit_transform(phrases))
    stage_one = _greedy_group(freq, char_matrix, char_threshold)
    labels = np.array([phrases[g[np.argmax(freq[g])]] for g in stage_one])
    totals = np.array([freq[g].sum() for g in stage_one])
    sizes = np.array([len(g) for g in stage_one])

    stripped = [" ".join(w for w in label.split()
                         if w not in FILLER_WORDS and len(w) > 1) or label
                for label in labels]
    word_matrix = normalize(TfidfVectorizer(analyzer="word").fit_transform(stripped))
    stage_two = _greedy_group(totals, word_matrix, word_threshold)

    rows = [{"complaint": labels[g[np.argmax(totals[g])]],
             "count": int(totals[g].sum()),
             "variants": int(sizes[g].sum())} for g in stage_two]
    return (pd.DataFrame(rows).sort_values("count", ascending=False)
            .reset_index(drop=True))


def scope_ids(scope: str) -> pd.Series:
    """Product ids for a scope, whether it names a category or a department."""
    column = "sub_category" if scope in SUB_CATEGORY_ALIASES else "Category1"
    return products.loc[products[column] == scope, "id"]


_COMPLAINT_CACHE: dict[str, pd.DataFrame] = {}


def complaint_counts(category: str) -> pd.Series:
    """Raw frequency of every complaint phrase written in a category."""
    ids = scope_ids(category)
    subset = comments[comments["product_id"].isin(ids)]
    cleaned = (subset["disadvantages"].dropna().astype(str)
               # The source stores list reprs, so backslash-r survives as text.
               .str.replace(r"\\r|\\n|[\[\]']", " ", regex=True)
               .str.split(",").explode()
               .map(normalize_text))
    cleaned = cleaned[(cleaned.str.len() > 2) & (~cleaned.isin(NON_COMPLAINTS))]
    return cleaned.value_counts()


def category_complaints(category: str, top_n: int = 15) -> pd.DataFrame:
    """Top complaints of a category, after merging equivalent phrasings.

    Memoised: a single analytics request needs this table twice, once for the
    console and once inside the prompt, and on a department-wide scope each
    computation is measured in minutes rather than seconds.
    """
    if category not in _COMPLAINT_CACHE:
        _COMPLAINT_CACHE[category] = cluster_complaints(complaint_counts(category))
    return _COMPLAINT_CACHE[category].head(top_n)


# Departments are stored as English slugs but managers name them in Persian.
SUB_CATEGORY_ALIASES = {
    "beauty": ("ارایشی", "بهداشتی", "زیبایی", "لوازم ارایش"),
    "clothe": ("پوشاک", "لباس", "مد"),
    "book & stationary & art": ("کتاب", "لوازم التحریر", "نوشت افزار", "هنر"),
    "toys and kids": ("اسباب بازی", "کودک", "بچه"),
    "travel": ("سفر", "کمپینگ"),
    "rural goods": ("کالای روستایی", "روستایی"),
}


def resolve_scope(text: str) -> tuple[str, str] | None:
    """Map a phrase to a catalogue scope: ('Category1'|'sub_category', value).

    Category names are checked first because they are the narrower, more
    specific reading of what the user asked for.
    """
    asked = {t for t in QUERY_TOKEN.findall(text) if strip_zwnj(t) not in INTENT_WORDS}
    if not asked:
        return None
    best, best_score = None, 0.0
    for name, words in CATEGORY_TOKENS.items():
        score = len(asked & words) / len(words)
        if score > best_score:
            best, best_score = name, score
    if best_score >= 0.6:
        return ("Category1", best)
    for slug, aliases in SUB_CATEGORY_ALIASES.items():
        if any(normalize_text(alias) in text for alias in aliases):
            return ("sub_category", slug)
    return None


def underperforming_products(category: str, min_reviews: int = 30, top_n: int = 10) -> pd.DataFrame:
    """Products with plenty of reviews but a low recommendation rate."""
    subset = products[products["id"].isin(scope_ids(category))
                      & (products["n_reviews"] >= min_reviews)]
    out = subset.nsmallest(top_n, "rec_score")[
        ["id", "title_fa", "n_reviews", "rec_ratio", "rec_score", "Price"]].copy()
    out["price_toman"] = out["Price"] // RIAL_PER_TOMAN
    return out.drop(columns=["Price"])


def brand_comparison(category: str, min_products: int = 5, top_n: int = 10) -> pd.DataFrame:
    """Compare the main brands of a category on satisfaction and price."""
    subset = products[products["id"].isin(scope_ids(category))
                      & (products["Brand"] != PLACEHOLDER_BRAND)]
    grouped = subset.groupby("Brand").agg(
        products=("id", "size"),
        reviews=("n_reviews", "sum"),
        rec_score=("rec_score", "mean"),
        median_price=("Price", "median"),
    )
    grouped = grouped[grouped["products"] >= min_products]
    grouped["median_price_toman"] = grouped["median_price"] // RIAL_PER_TOMAN
    return (grouped.drop(columns=["median_price"])
            .sort_values("reviews", ascending=False).head(top_n).round(3))


CATEGORY = products.loc[PRODUCT_ROW[demo_id], "Category1"]
print(f"دسته تحلیل‌شده: {CATEGORY}\n")

# Before/after, because the effect of clustering is the point worth showing.
raw = complaint_counts(CATEGORY)
complaints = category_complaints(CATEGORY)
print(f"عبارت‌های یکتای شکایت: {len(raw):,}  ->  خوشه‌ها: "
      f"{len(cluster_complaints(raw)):,}\n")
print("--- بدون خوشه‌بندی (شمارش خام) ---")
print(pd.DataFrame({"complaint": raw.index[:8], "count": raw.to_numpy()[:8]})
      .to_string(index=False))
print("\n--- با خوشه‌بندی ---")
print(complaints.to_string(index=False))
print("\n--- محصولات پرنظر با پیشنهاد خرید پایین ---")
print(underperforming_products(CATEGORY).to_string(index=False))
print("\n--- مقایسه برندها ---")
print(brand_comparison(CATEGORY).to_string())

دسته تحلیل‌شده: اصلاح موی صورت

عبارت‌های یکتای شکایت: 4,628  ->  خوشه‌ها: 2,436

--- بدون خوشه‌بندی (شمارش خام) ---
  complaint  count
صفر زن نیست     81
  قیمت بالا     66
  صدای زیاد     60
 صفر نمیزنه     42
 موتور ضعیف     36
       قیمت     30
 خط زن ضعیف     26
    قدرت کم     26

--- با خوشه‌بندی ---
       complaint  count  variants
     صفر زن نیست    265        71
       قیمت بالا    157        41
       صدای زیاد    137        43
      خط زن ضعیف     97        38
     کیفیت پایین     96        52
      موتور ضعیف     93        30
         قدرت کم     86        35
    نداشتن شارژر     80        31
      نداشتن کیف     71        22
      باتری ضعیف     54        20
        تیغه کند     52        25
  نداشتن آداپتور     50        27
       بدنه ضعیف     47        29
زمان شارژ طولانی     43        28
     ضد آب نبودن     42        15

--- محصولات پرنظر با پیشنهاد خرید پایین ---
     id                                          title_fa  n_reviews  rec_ratio  rec_score  price_tom

In [12]:
ANALYTICS_PROMPT = """تو تحلیلگر یک فروشگاه اینترنتی هستی و برای مدیر دسته «{category}» گزارش می‌نویسی.

## پرتکرارترین شکایت‌های ثبت‌شده کاربران
{complaints}

## محصولات پرنظر با پایین‌ترین درصد پیشنهاد خرید
{weak}

## مقایسه برندهای اصلی
{brands}

## قواعد
- فقط بر پایه اعداد بالا بنویس و عدد جدیدی نساز.
- سه یافته اصلی و برای هرکدام یک اقدام پیشنهادی بنویس.
- پاسخ فارسی، حداکثر ۲۰۰ کلمه."""


def category_report(category: str) -> dict:
    """Capability 4: a manager-facing summary built on precomputed aggregates."""
    started = time.time()
    prompt = ANALYTICS_PROMPT.format(
        category=category,
        complaints=category_complaints(category).to_string(index=False),
        weak=underperforming_products(category).to_string(index=False),
        brands=brand_comparison(category).to_string(),
    )
    answer = ask_gemini(prompt, tag="category-analytics", max_tokens=800)
    return {"category": category, "answer": answer.strip(),
            "latency_s": round(time.time() - started, 2)}


report = category_report(CATEGORY)
print(f"{'='*70}\nگزارش دسته: {report['category']}\n{'='*70}")
print(report["answer"])
print(f"\n({report['latency_s']}s)")

گزارش دسته: اصلاح موی صورت
به عنوان تحلیلگر فروشگاه، گزارش وضعیت دسته «اصلاح موی صورت» بر اساس داده‌های موجود به شرح زیر است:

**یافته‌های اصلی و اقدامات پیشنهادی:**

۱. **بزرگترین نارضایتی کاربران مربوط به کارایی محصول است.** شکایت «صفرزن نیست» با ۲۶۵ مورد و ۷۱ مدل، پرتکرارترین نارضایتی است. همچنین ضعف خط‌زن (۹۷ مورد)، موتور (۹۳ مورد) و قدرت کم (۸۶ مورد) از مشکلات عمده‌اند.
*   **اقدام پیشنهادی:** در صفحه محصولات پرفروش، مشخصات فنی (قدرت موتور و قابلیت اصلاح صفر) با دقت شفاف‌سازی شود تا انتظارات مشتریان مدیریت گردد.

۲. **محصولات فیلیپس با وجود قیمت بالا، امتیاز پیشنهاد خرید پایینی دارند.** مدل HQ7320 فیلیپس با قیمت ۸,۴۰۰,۰۰۰ تومان، نرخ پیشنهاد خرید **۰.۰** دارد. در مقابل، برندهای اقتصادی مثل «وی جی ار» و «کیمی» با تعداد نظرات بالا (به‌ترتیب ۶۳۷۴ و ۴۳۶۰ نظیر)، امتیازات رضایت مناسبی (۰.۷۸۵ و ۰.۷۳۸) کسب کرده‌اند.
*   **اقدام پیشنهادی:** بازنگری در سبد کالاهای رده‌بالای فیلیپس و تمرکز بیشتر روی تامین برندهای محبوب و پربازخورد اقتصادی مانند «وی جی ار» و «پاناسونیک» (با بالاترین امتیاز ۰.۸

## ۹. ارزیابی — بخش چهارم پروژه

چهار چیز اندازه‌گیری می‌شود:

- **Retrieval Quality** روی یک مجموعه پرس‌وجوی دستی‌ساخته، با مقایسه دو نسخه لایه فهم پرس‌وجو
- **Grounding** یعنی چند درصد شناسه‌های مورد استناد واقعاً در شواهد داده‌شده بوده‌اند
- **Latency** به تفکیک مرحله
- **Cost** به تفکیک قابلیت

In [13]:
# Hand-built evaluation set: each query names the category a correct answer
# must fall into. Small on purpose - it is labelled by hand.
# The first four strings are byte-identical to EXAMPLE_QUERIES, so the LLM
# variant answers them straight from the cache and costs nothing extra.
EVAL_QUERIES = [
    (EXAMPLE_QUERIES[0], "کیف"),
    (EXAMPLE_QUERIES[1], "سفر|کمپینگ|چمدان|ساک|کوله"),
    (EXAMPLE_QUERIES[2], "آبرسان|مرطوب"),
    (EXAMPLE_QUERIES[3], "فکری|آموزشی|پازل|بازی"),
    ("ماگ سفری استیل", "ماگ|لیوان|فلاسک"),
    ("دفتر یادداشت با جلد سخت", "دفتر|یادداشت"),
]


def precision_at_k(plan_fn, k: int = 5) -> pd.DataFrame:
    """Fraction of the top-k results whose title matches the expected pattern."""
    rows = []
    for query, pattern in EVAL_QUERIES:
        started = time.time()
        plan = plan_fn(query)
        results = search_products(plan, top_k=k)
        elapsed = time.time() - started
        if results.empty:
            hits = 0
        else:
            hits = int(results["title_fa"].str.contains(pattern, regex=True, na=False).sum())
        rows.append({"query": query[:40], "concept": plan["concept"][:28],
                     "hits": hits, "p@k": hits / k, "seconds": round(elapsed, 2)})
    return pd.DataFrame(rows)


rules_eval = precision_at_k(understand_query_rules)
llm_eval = precision_at_k(understand_query_llm)
print("--- baseline: rule-based query understanding ---")
print(rules_eval.to_string(index=False))
print(f"mean P@5 = {rules_eval['p@k'].mean():.3f} | mean latency = {rules_eval['seconds'].mean():.2f}s")
print("\n--- with LLM query understanding ---")
print(llm_eval.to_string(index=False))
print(f"mean P@5 = {llm_eval['p@k'].mean():.3f} | mean latency = {llm_eval['seconds'].mean():.2f}s")
print(f"\nimprovement: {llm_eval['p@k'].mean() - rules_eval['p@k'].mean():+.3f}")

--- baseline: rule-based query understanding ---
                                   query                   concept  hits  p@k  seconds
یک کیف برای استفاده روزمره می‌خوام که خی                       کیف     5  1.0     0.05
چند وسیله کاربردی برای سفر معرفی کن که ن                       سفر     4  0.8     0.06
      کرم آبرسان صورت زیر ۲۰۰ هزار تومان           کرم آبرسان صورت     5  1.0     0.10
     یک اسباب بازی فکری برای کودک ۵ ساله اسباب بازی فکری کودک ساله     5  1.0     0.16
                          ماگ سفری استیل            ماگ سفری استیل     5  1.0     0.11
                 دفتر یادداشت با جلد سخت      دفتر یادداشت جلد سخت     5  1.0     0.14
mean P@5 = 0.967 | mean latency = 0.10s

--- with LLM query understanding ---
                                   query                     concept  hits  p@k  seconds
یک کیف برای استفاده روزمره می‌خوام که خی                  کیف روزمره     5  1.0     0.08
چند وسیله کاربردی برای سفر معرفی کن که ن           وسایل کاربردی سفر     0  0.0     0.

In [14]:
# Grounding: every id the model cites must come from the evidence it was given.
# Two products x two questions = four calls. Small on purpose: the free tier
# allows only 20 requests per model per day, which is the binding constraint
# on this project - not the dollar budget.
GROUNDING_QUESTIONS = [
    "مردم بیشتر از چه چیزی راضی بودند؟",
    "ایرادهای پرتکرار این محصول چیست؟",
]
sample_products = products[products["evidence_tier"] == "rich"].nlargest(2, "n_substantive")["id"].tolist()

rows = []
for pid in sample_products:
    for question in GROUNDING_QUESTIONS:
        result = answer_about_product(pid, question, top_k=15)
        given = set(result["evidence_ids"])
        cited = set(result["cited_ids"])
        rows.append({
            "product": result["product"][:32],
            "question": question[:26],
            "tier": result["evidence_tier"],
            "n_evidence": len(given),
            "n_cited": len(cited),
            "valid_citations": len(cited & given),
            "hallucinated": len(cited - given),
            "latency_s": result["latency_s"],
        })

grounding = pd.DataFrame(rows)
print(grounding.to_string(index=False))
total_cited = grounding["n_cited"].sum()
valid = grounding["valid_citations"].sum()
print(f"\nGrounding: {valid}/{total_cited} citations resolve to supplied evidence "
      f"({valid / max(total_cited, 1) * 100:.1f}%)")
print(f"answers with at least one citation: "
      f"{(grounding['n_cited'] > 0).mean() * 100:.0f}%")
print(f"mean answer latency: {grounding['latency_s'].mean():.2f}s")

                         product                   question tier  n_evidence  n_cited  valid_citations  hallucinated  latency_s
تصفیه آب خانگی اس اس وی مدل MaxT مردم بیشتر از چه چیزی راضی rich          15        8                8             0        0.0
تصفیه آب خانگی اس اس وی مدل MaxT ایرادهای پرتکرار این محصول rich          15        4                4             0        0.0
دستگاه تصفیه کننده آب خانگی اولا مردم بیشتر از چه چیزی راضی rich          15        4                4             0        0.0
دستگاه تصفیه کننده آب خانگی اولا ایرادهای پرتکرار این محصول rich          15        3                3             0        0.0

Grounding: 19/19 citations resolve to supplied evidence (100.0%)
answers with at least one citation: 100%
mean answer latency: 0.00s


In [15]:
print("=== هزینه و مصرف توکن ===")
report = cost_report()
print(report.to_string())

total = report.loc["TOTAL"]
print(f"\nمجموع فراخوانی‌ها : {int(total['calls'])}")
print(f"توکن ورودی        : {int(total['input_tokens']):,}")
print(f"توکن خروجی        : {int(total['output_tokens']):,}")
print(f"هزینه تخمینی      : ${total['usd']:.4f}  از بودجه ۵ دلاری")
print(f"کش                : {len(CACHE)} پاسخ ذخیره‌شده (اجرای دوباره رایگان است)")

report.to_csv(ARTIFACT_DIR / "cost_report.csv")
grounding.to_csv(ARTIFACT_DIR / "grounding_eval.csv", index=False)
rules_eval.to_csv(ARTIFACT_DIR / "retrieval_rules.csv", index=False)
llm_eval.to_csv(ARTIFACT_DIR / "retrieval_llm.csv", index=False)
print("\nsaved: cost_report.csv, grounding_eval.csv, retrieval_rules.csv, retrieval_llm.csv")

=== هزینه و مصرف توکن ===
                     calls  input_tokens  output_tokens  thought_tokens     usd  mean_latency_s
tag                                                                                            
category-analytics     7.0        7837.0         2732.0             0.0  0.0092          4.3056
comparison             7.0       18692.0         2401.0             0.0  0.0116          5.6200
product-qa            23.0       34525.0         4948.0             0.0  0.0227          8.3573
query-understanding   17.0        3722.0          468.0             0.0  0.0023          6.2526
smoke-test             2.0          24.0            1.0             0.0  0.0000          8.8695
TOTAL                 56.0       64800.0        10550.0             0.0  0.0458          6.8881

مجموع فراخوانی‌ها : 56
توکن ورودی        : 64,800
توکن خروجی        : 10,550
هزینه تخمینی      : $0.0458  از بودجه ۵ دلاری
کش                : 32 پاسخ ذخیره‌شده (اجرای دوباره رایگان است)

saved: cost_repor

## ۱۰. رابط چت

چهار قابلیت بالا هرکدام یک تابع جداگانه‌اند. اینجا یک لایه مسیریابی روی آن‌ها
گذاشته می‌شود تا کاربر فقط سؤالش را بپرسد و سیستم خودش تصمیم بگیرد کدام قابلیت
را صدا بزند.

مسیریابی **قاعده‌محور** است نه با مدل زبانی، به دو دلیل: تأخیر صفر، و مهم‌تر
اینکه سهمیه رایگان روزی ۲۰ درخواست است و صرف کردن یکی از آن‌ها برای تشخیص نیت
اتلاف است.

مستندات هر پاسخ در پنل پایین قابل مشاهده است — هم نظراتی که به مدل داده شد و هم
اینکه مدل به کدامشان استناد کرده.

In [16]:
from IPython.display import HTML, display, clear_output

# Intent comes from two signals, not one: what is being asked, and at what
# scope. "پرتکرارترین ایراد این محصول" and "پرتکرارترین ایراد این دسته" share
# every keyword and differ only in scope, so scope is checked first.
COMPARE_WORDS = ("مقایسه", "کدوم بهتر", "کدام بهتر", "فرق", "تفاوت", "بهتره", "بهتر است")
# "دسته" needs a preposition or determiner in front of it to mean a catalogue
# category. A bare match would send the product search "دسته بازی پلی استیشن"
# to the analytics branch.
CATEGORY_SCOPE_PATTERN = re.compile(
    r"(این|کل|همین|در|برای|از|سراسر)\s*‌?\s*دسته"
    r"|دسته‌?\s*بندی|برندها|برند‌ها|مدیر|بازار"
    r"|گزارش دسته|تحلیل دسته|کدام محصولات|کدوم محصولا")
PRODUCT_SCOPE = ("این محصول", "این کالا", "این یکی", "همین محصول", "همین کالا")
QA_WORDS = ("راضی", "ایراد", "مشکل", "کیفیت", "ارزش خرید", "تجربه", "چطوره",
            "چطور است", "نظر کاربران", "خریداران", "معایب", "مزایا", "شکایت",
            "نارضایتی", "پرتکرار")
ORDINALS = {"اولی": 1, "اولین": 1, "دومی": 2, "دومین": 2, "سومی": 3, "سومین": 3,
            "چهارمی": 4, "پنجمی": 5}

# Tokenised once: 228 category names, checked against every analytics question.
CATEGORY_TOKENS = {
    name: {t for t in QUERY_TOKEN.findall(normalize_text(name))
           if strip_zwnj(t) not in INTENT_WORDS}
    for name in products["Category1"].dropna().unique()
}
CATEGORY_TOKENS = {k: v for k, v in CATEGORY_TOKENS.items() if v}
print(f"category vocabulary: {len(CATEGORY_TOKENS)} names")


class ShoppingAssistant:
    """Routes one Persian question to the right capability and remembers context.

    State is what turns four isolated functions into a conversation: after a
    search the user can say "دومی" or "ایرادهاش چیه" without naming the product
    again.
    """

    def __init__(self, page_size: int = 5):
        self.page_size = page_size
        self.results = products.head(0)
        self.selected = None

    # -- routing -----------------------------------------------------------
    def _pick_index(self, text: str) -> int | None:
        """Read a 1-based reference to a row of the last result set."""
        for word, number in ORDINALS.items():
            if word in text:
                return number
        match = re.search(r"\b([1-9])\b", text)
        return int(match.group(1)) if match else None

    def _match_category(self, text: str) -> str | None:
        """The catalogue scope the question names, if any.

        Only a Category1 hit counts as a routing signal. Department aliases
        such as "کتاب" are far too common in ordinary product searches to be
        allowed to divert a query into the analytics branch on their own.
        """
        found = resolve_scope(text)
        return found[1] if found and found[0] == "Category1" else None

    def _rank_by_name(self, text: str) -> list[tuple[int, float]]:
        """Score every listed product against the words of the question.

        Shoppers refer to a result by its title, not by its row number. The
        score is the share of a title's own words that the question repeats,
        so a long title is not penalised for the words it adds.
        """
        if self.results.empty:
            return []
        asked = {t for t in QUERY_TOKEN.findall(text)
                 if strip_zwnj(t) not in INTENT_WORDS}
        if not asked:
            return []
        scored = []
        for position, title in enumerate(self.results["title_norm"].fillna(""), start=1):
            title_words = set(QUERY_TOKEN.findall(title))
            if title_words:
                scored.append((position, len(asked & title_words) / len(title_words)))
        return sorted(scored, key=lambda pair: -pair[1])

    def _match_by_name(self, text: str, threshold: float = 0.4) -> int | None:
        """The single product the question names, if one stands out."""
        ranked = self._rank_by_name(text)
        return ranked[0][0] if ranked and ranked[0][1] >= threshold else None

    def route(self, text: str) -> str:
        has_results = not self.results.empty
        index = self._pick_index(text)
        # Naming a catalogue category is itself a category-scope signal.
        category_scope = bool(CATEGORY_SCOPE_PATTERN.search(text)) or bool(
            self._match_category(text))
        product_scope = any(w in text for w in PRODUCT_SCOPE)

        # Scope is decided first: the same keyword serves a single product and a
        # whole category, and only the scope tells them apart.
        if category_scope and not product_scope:
            return "analytics"
        if any(w in text for w in COMPARE_WORDS) and has_results:
            return "compare"
        if any(w in text for w in QA_WORDS):
            return "qa" if (self.selected or has_results) else "search"
        if index and has_results and len(text.split()) <= 3:
            return "select"
        return "search"

    # -- capabilities ------------------------------------------------------
    def ask(self, query: str) -> dict:
        text = normalize_text(query)
        intent = self.route(text)
        started = time.time()
        try:
            payload = getattr(self, f"_do_{intent}")(query, text)
        except Exception as error:                       # quota, network, parsing
            payload = {"kind": "error",
                       "text": f"خطا در پردازش: {type(error).__name__} — {str(error)[:160]}"}
        payload.setdefault("evidence", comments.head(0))
        payload.setdefault("cited", [])
        payload["intent"] = intent
        payload["latency"] = round(time.time() - started, 2)
        return payload

    def _do_search(self, query: str, text: str) -> dict:
        plan = understand_query_rules(query)
        self.results = search_products(plan, top_k=self.page_size)
        self.selected = int(self.results.iloc[0]["id"]) if not self.results.empty else None
        if self.results.empty:
            return {"kind": "search", "text": "محصولی با این مشخصات پیدا نشد.", "plan": plan}
        return {"kind": "search", "plan": plan, "products": self.results,
                "text": f"{len(self.results)} محصول پیدا شد. برای پرسیدن درباره هرکدام "
                        f"شماره‌اش را بنویسید، یا مستقیم سؤالتان را بپرسید."}

    def _do_select(self, query: str, text: str) -> dict:
        index = self._pick_index(text) or 1
        if index > len(self.results):
            return {"kind": "error", "text": f"فقط {len(self.results)} نتیجه در فهرست هست."}
        row = self.results.iloc[index - 1]
        self.selected = int(row["id"])
        facts = product_facts(self.selected)
        return {"kind": "select", "products": self.results.iloc[[index - 1]],
                "text": f"محصول «{facts['title']}» انتخاب شد. "
                        f"{facts['n_reviews']} نظر دارد و سطح شواهدش {facts['evidence_tier']} است."}

    def _do_qa(self, query: str, text: str) -> dict:
        # An explicit number wins; otherwise fall back to naming the product.
        index = self._pick_index(text) or self._match_by_name(text)
        if index and index <= len(self.results):
            self.selected = int(self.results.iloc[index - 1]["id"])
        if self.selected is None:
            return {"kind": "error", "text": "اول یک محصول را جست‌وجو یا انتخاب کنید."}
        result = answer_about_product(self.selected, query, top_k=15)
        evidence = get_evidence(self.selected, query, top_k=15)
        return {"kind": "qa", "text": result["answer"], "evidence": evidence,
                "cited": result["cited_ids"], "product": result["product"],
                "tier": result["evidence_tier"]}

    def _do_compare(self, query: str, text: str) -> dict:
        if len(self.results) < 2:
            return {"kind": "error", "text": "برای مقایسه حداقل دو نتیجه لازم است."}

        # A comparison question names two products, so each one's words are
        # diluted across the sentence; the bar is lower than for a single
        # product. Positions the user asked for beat the first two rows.
        named = [position for position, score in self._rank_by_name(text) if score >= 0.3]
        explicit = [n for n in (self._pick_index(text),) if n]
        positions = (explicit + [p for p in named if p not in explicit])[:2]
        if len(positions) < 2:
            positions = [1, 2]

        chosen = self.results.iloc[[p - 1 for p in positions]]
        ids = chosen["id"].astype(int).tolist()
        result = compare_products(ids, query)
        rows = [i for group in result["evidence_ids"].values() for i in group]
        return {"kind": "compare", "text": result["answer"],
                "evidence": comments[comments["id"].isin(rows)],
                "cited": result["cited_ids"], "products": chosen}

    def _do_analytics(self, query: str, text: str) -> dict:
        # A category the user names outranks the category of whatever product
        # happened to be selected earlier in the conversation.
        found = resolve_scope(text)
        category = found[1] if found else None
        if category is None and self.selected is not None:
            category = products.loc[PRODUCT_ROW[self.selected], "Category1"]
        if category is None:
            plan = understand_query_rules(query)
            found = search_products(plan, top_k=1)
            if found.empty:
                return {"kind": "error",
                        "text": "دسته مورد نظر پیدا نشد. نام دسته را بنویسید یا اول محصولی جست‌وجو کنید."}
            category = found.iloc[0]["Category1"]
        report = category_report(category)
        return {"kind": "analytics", "text": report["answer"], "category": category,
                "product": f"دسته «{category}»",
                "complaints": category_complaints(category, top_n=10)}


assistant = ShoppingAssistant()
print("assistant ready | intents:", ", ".join(("search", "select", "qa", "compare", "analytics")))

category vocabulary: 195 names
assistant ready | intents: search, select, qa, compare, analytics


In [17]:
# --- rendering --------------------------------------------------------
RTL = "direction:rtl;text-align:right;font-family:Tahoma,Arial,sans-serif;"
INTENT_LABEL = {"search": "جست‌وجوی محصول", "select": "انتخاب محصول",
                "qa": "پرسش و پاسخ روی نظرات", "compare": "مقایسه محصولات",
                "analytics": "تحلیل دسته", "error": "خطا"}


def product_cards(frame: pd.DataFrame) -> str:
    """Numbered product rows, so the user can refer to one by its number."""
    rows = []
    for number, (_, row) in enumerate(frame.iterrows(), start=1):
        rows.append(
            f"<div style='{RTL}background:#fbf7f5;border:1px solid #e8dfdc;"
            f"border-radius:8px;padding:8px 12px;margin:6px 0'>"
            f"<b style='color:#c41e3a'>{number}.</b> {row['title_fa']}<br>"
            f"<span style='color:#6b7280;font-size:12px'>"
            f"{row['price_toman']:,} تومان · {row['n_reviews']} نظر · "
            f"رضایت {row['rec_score']:.2f} · شواهد {row['evidence_tier']}</span></div>")
    return "".join(rows)


def evidence_table(evidence: pd.DataFrame, cited: list[int]) -> str:
    """Every review handed to the model, with the cited ones marked."""
    if evidence.empty:
        return f"<div style='{RTL}color:#6b7280'>این پاسخ از نظرات کاربران استفاده نکرده است.</div>"
    cited_set = set(cited)
    rows = []
    for _, row in evidence.iterrows():
        used = int(row["id"]) in cited_set
        mark = ("<span style='background:#c41e3a;color:#fff;border-radius:4px;"
                "padding:1px 6px;font-size:11px'>استناد شد</span>" if used else
                "<span style='color:#a0aec0;font-size:11px'>داده شد</span>")
        body = str(row["body"]).replace("<", "&lt;").replace("\n", " ")
        extra = ""
        if pd.notna(row["disadvantages"]):
            extra += f"<br><span style='color:#c41e3a;font-size:11px'>منفی: {row['disadvantages']}</span>"
        rows.append(
            f"<div style='{RTL}border-bottom:1px solid #eee;padding:8px 4px'>"
            f"<code style='color:#2f3c7e'>[{row['id']}]</code> {mark} "
            f"<span style='color:#6b7280;font-size:11px'>امتیاز {row['rate']}</span><br>"
            f"{body}{extra}</div>")
    header = (f"<div style='{RTL}padding:6px 4px;font-weight:bold'>"
              f"{len(evidence)} نظر به مدل داده شد · {len(cited_set)} مورد استناد قرار گرفت</div>")
    return header + "".join(rows)


def render_turn(query: str, payload: dict) -> str:
    """One question and its answer as an HTML block."""
    blocks = [
        f"<div style='{RTL}background:#141b2d;color:#fff;border-radius:10px;"
        f"padding:10px 14px;margin:10px 0 4px 0'>{query}</div>",
        f"<div style='{RTL}color:#7a8290;font-size:11px;margin-bottom:4px'>"
        f"{INTENT_LABEL.get(payload['intent'], payload['intent'])} · "
        f"{payload['latency']} ثانیه</div>",
    ]
    # Naming the product the answer is about makes a mis-selection obvious
    # instead of leaving the reader to spot it inside the prose.
    if payload.get("product"):
        blocks.append(
            f"<div style='{RTL}background:#fbf7f5;border:1px solid #e8dfdc;"
            f"border-radius:8px;padding:6px 12px;margin-bottom:6px;font-size:13px'>"
            f"<b style='color:#c41e3a'>درباره:</b> {payload['product']}"
            f"<span style='color:#6b7280'> · سطح شواهد {payload.get('tier', '')}</span></div>")
    if payload.get("products") is not None and not payload["products"].empty:
        blocks.append(product_cards(payload["products"]))
    answer = payload["text"].replace("\n", "<br>")
    blocks.append(
        f"<div style='{RTL}background:#fff;border:1px solid #e8dfdc;border-radius:10px;"
        f"padding:12px 14px;line-height:1.9'>{answer}</div>")
    return "".join(blocks)


# --- widgets ----------------------------------------------------------
# widgets.Output scrolls unreliably in VS Code and jumps back to the top on
# every redraw. widgets.HTML with its own scrolling <div> behaves correctly,
# and keeping the newest answer in an unbounded panel of its own means the
# thing you just asked about never needs scrolling to read.
import ipywidgets as widgets


def scroll_box(inner_html: str, height: int = 320) -> str:
    return (f"<div style='max-height:{height}px;overflow-y:auto;"
            f"border:1px solid #e8dfdc;border-radius:8px;padding:8px'>{inner_html}</div>")


WELCOME = (f"<div style='{RTL}color:#6b7280;line-height:2'>"
           "<b>نمونه سؤال‌ها:</b><br>"
           "· یک کیف برای استفاده روزمره که خیلی گرون نباشه<br>"
           "· ایرادهای پرتکرار محصول ۲ چیست؟<br>"
           "· این دو را مقایسه کن<br>"
           "· پرتکرارترین شکایت این دسته چیست؟</div>")

latest_panel = widgets.HTML(value=WELCOME)
history_panel = widgets.HTML(value=f"<div style='{RTL}color:#a0aec0'>هنوز گفت‌وگویی نشده.</div>")
evidence_panel = widgets.HTML(value=f"<div style='{RTL}color:#a0aec0'>هنوز پاسخی داده نشده.</div>")

history_box = widgets.Accordion(children=[history_panel])
history_box.set_title(0, "تاریخچه گفت‌وگو")
history_box.selected_index = None            # collapsed; the latest is already above

evidence_box = widgets.Accordion(children=[evidence_panel])
evidence_box.set_title(0, "مستندات پاسخ آخر")
evidence_box.selected_index = 0

question_input = widgets.Text(
    placeholder="سؤالتان را بنویسید و Enter بزنید...",
    layout=widgets.Layout(width="78%"))
send_button = widgets.Button(description="بپرس", button_style="danger",
                             layout=widgets.Layout(width="10%"))
reset_button = widgets.Button(description="شروع دوباره",
                              layout=widgets.Layout(width="10%"))
transcript = []


def handle(_=None):
    query = question_input.value.strip()
    if not query:
        return
    question_input.value = ""
    send_button.description = "..."
    try:
        payload = assistant.ask(query)
    finally:
        send_button.description = "بپرس"

    block = render_turn(query, payload)
    latest_panel.value = block
    transcript.insert(0, block)              # newest first, so history opens on it
    history_box.set_title(0, f"تاریخچه گفت‌وگو ({len(transcript)} پرسش)")
    history_panel.value = scroll_box("<hr style='border:none;border-top:1px solid #eee'>"
                                     .join(transcript), 360)
    evidence_panel.value = scroll_box(
        evidence_table(payload["evidence"], payload["cited"]), 320)
    evidence_box.set_title(0, f"مستندات پاسخ آخر ({len(payload['evidence'])} نظر)")


def reset(_=None):
    transcript.clear()
    assistant.results = products.head(0)
    assistant.selected = None
    latest_panel.value = WELCOME
    history_panel.value = f"<div style='{RTL}color:#a0aec0'>هنوز گفت‌وگویی نشده.</div>"
    evidence_panel.value = f"<div style='{RTL}color:#a0aec0'>هنوز پاسخی داده نشده.</div>"
    history_box.set_title(0, "تاریخچه گفت‌وگو")
    evidence_box.set_title(0, "مستندات پاسخ آخر")


send_button.on_click(handle)
reset_button.on_click(reset)
question_input.on_submit(handle)

display(widgets.VBox([
    widgets.HBox([send_button, reset_button, question_input]),
    latest_panel,
    evidence_box,
    history_box,
]))

C:\Users\M.Ali\AppData\Local\Temp\ipykernel_21728\761532612.py:150: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  question_input.on_submit(handle)
